In [ ]:
%pip install captum
!git clone https://github.com/kapitan05/causal_calibration.git
%cd causal_calibration
%mkdir -p data

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from tqdm import tqdm
import random

import torch.nn as nn
import torchvision.models as models
from torchvision import transforms

from datasets import load_dataset, Image as HFImage
from google.colab import userdata

from src.models import get_preprocessing_transforms
from src.attribution import AttributionPipeline
from src.causal_tests import evaluate_causal_metric
from src.calibration import ReCalXModel, temperature_scaling
from src.generators import BucketDeletionGenerator
from src.metrics import calculate_ece, calculate_tace

In [ ]:
def normalize_saliency(sal: np.ndarray) -> np.ndarray:
    sal = np.maximum(sal, 0)
    return (sal - sal.min()) / (sal.max() - sal.min() + 1e-8)

def denormalize(tensor: torch.Tensor) -> np.ndarray:
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
    return np.clip(img * std + mean, 0, 1)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Raw logits — required for temperature scaling and ReCalXModel
raw_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT).to(device).eval()
for p in raw_model.parameters():
    p.requires_grad = False

# With softmax — used only for top-1 class selection
model_softmax = nn.Sequential(raw_model, nn.Softmax(dim=1)).eval()

def get_top1(tensor: torch.Tensor) -> int:
    with torch.no_grad():
        return int(torch.argmax(model_softmax(tensor.to(device))).item())

print("ResNet50 loaded.")

In [ ]:
# Same code path used by collect_logits_for_calibration in src/calibration.py
pipeline = AttributionPipeline(raw_model)
print("AttributionPipeline ready.")

In [ ]:
k = userdata.get("hf")
!wget -q --header="Authorization: Bearer {k}" \
  "https://huggingface.co/datasets/ILSVRC/imagenet-1k/resolve/main/data/train-00000-of-00294.parquet"

hf_dataset = load_dataset("parquet", data_files="train-00000-of-00294.parquet", split="train")
hf_dataset = hf_dataset.cast_column("image", HFImage())
print(f"Loaded {len(hf_dataset)} images.")

In [ ]:
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); np.random.seed(SEED)

preprocess = get_preprocessing_transforms()
indices  = random.sample(range(len(hf_dataset)), 20)
selected = hf_dataset.select(indices)

samples = []
for item in tqdm(selected, desc="Loading images"):
    pil_img = item["image"].convert("RGB")
    tensor  = preprocess(pil_img).unsqueeze(0)   # (1, 3, 224, 224)
    samples.append((pil_img, tensor))

img_batch      = torch.cat([t for _, t in samples], dim=0)   # (20, 3, 224, 224)
target_classes = [get_top1(t) for _, t in samples]

print(f"img_batch: {img_batch.shape}")
print(f"Target classes: {target_classes}")

In [ ]:
rise_sals = []
for i, (_, tensor) in enumerate(tqdm(samples, desc="RISE saliency")):
    sal = pipeline.generate_map(tensor, target_classes[i], "rise").numpy()  # (224, 224)
    rise_sals.append(sal)

rise_sal_batch = np.stack(rise_sals)   # (20, 224, 224)
print(f"rise_sal_batch: {rise_sal_batch.shape}")

## Section 2 — Single Image Deep-Dive

Walk through one image in detail: build a deletion sequence, run both the raw and calibrated model, and inspect the fitted temperatures and confidence curves.

In [ ]:
IMG_IDX      = 0
single_tensor = img_batch[IMG_IDX:IMG_IDX+1]   # (1, 3, 224, 224)
target_class  = target_classes[IMG_IDX]

sal_norm   = normalize_saliency(rise_sals[IMG_IDX])   # [0, 1]
sal_tensor = torch.from_numpy(sal_norm)

gen_del = BucketDeletionGenerator(num_buckets=25)
seq_del, lvl_del = gen_del(single_tensor.cpu(), sal_tensor)

# Visualise original image + saliency overlay
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
ax1.imshow(denormalize(single_tensor)); ax1.axis("off"); ax1.set_title(f"Image {IMG_IDX} (class {target_class})")
ax2.imshow(denormalize(single_tensor))
ax2.imshow(sal_norm, cmap="jet", alpha=0.5); ax2.axis("off"); ax2.set_title("RISE saliency")
plt.tight_layout(); plt.show()

print(f"Sequence shape : {seq_del.shape}")
print(f"Levels (first5): {[f'{l:.2f}' for l in lvl_del[:5]]} ...")

In [ ]:
raw_probs, raw_auc, raw_all_probs = evaluate_causal_metric(
    raw_model, seq_del.to(device), lvl_del, target_class
)
print(f"Raw model | AUC: {raw_auc:.4f} | Initial confidence: {raw_probs[0]:.4f}")

In [ ]:
NUM_BINS = 10

logits_bins = {b: [] for b in range(NUM_BINS)}
labels_bins = {b: [] for b in range(NUM_BINS)}

with torch.no_grad():
    raw_logits_all = raw_model(seq_del.to(device)).cpu().numpy()   # (26, 1000)

for step_idx, logits in enumerate(raw_logits_all):
    level   = lvl_del[step_idx]
    bin_idx = min(int(level * NUM_BINS), NUM_BINS - 1)
    logits_bins[bin_idx].append(logits)
    labels_bins[bin_idx].append(target_class)

learned_temps_single = []
for b in range(NUM_BINS):
    lgt = logits_bins[b]
    lbl = labels_bins[b]
    if len(lgt) > 1:
        T = temperature_scaling(np.stack(lgt), np.array(lbl, dtype=np.int64))
    else:
        T = 1.0
    learned_temps_single.append(round(T, 4))

print(f"{'Bin':>4}  {'Level range':>12}  {'Samples':>8}  {'T':>8}")
print("-" * 40)
for b, T in enumerate(learned_temps_single):
    lo, hi = b / NUM_BINS, (b + 1) / NUM_BINS
    n = len(logits_bins[b])
    print(f"{b:>4}  {lo:.1f}–{hi:.1f}        {n:>8}  {T:>8.4f}")

In [ ]:
recalx_single = ReCalXModel(raw_model, num_bins=NUM_BINS)
recalx_single.load_learned_temperatures(learned_temps_single)

cal_probs, cal_auc, cal_all_probs = evaluate_causal_metric(
    recalx_single, seq_del.to(device), lvl_del, target_class
)
print(f"Calibrated model | AUC: {cal_auc:.4f} | Initial confidence: {cal_probs[0]:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bins_x = np.arange(NUM_BINS)
ax.bar(bins_x, learned_temps_single, color="steelblue", edgecolor="white", linewidth=0.5)
ax.axhline(1.0, color="red", linestyle="--", alpha=0.7, label="T=1 (no calibration)")
ax.set_xlabel("Perturbation bin")
ax.set_ylabel("Temperature T")
ax.set_title(f"Fitted temperatures — Image {IMG_IDX} (single-image, n≈2-3 per bin)")
ax.set_xticks(bins_x)
ax.set_xticklabels([f"{b/NUM_BINS:.1f}" for b in range(NUM_BINS)], rotation=45, fontsize=8)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lvl_del, raw_probs, "o-", color="steelblue",  label=f"Raw       (AUC={raw_auc:.3f})")
ax.plot(lvl_del, cal_probs, "s-", color="darkorange", label=f"Calibrated (AUC={cal_auc:.3f})")
ax.set_xlabel("Fraction of pixels deleted")
ax.set_ylabel(f"P(class={target_class})")
ax.set_title(f"Deletion confidence curve — Image {IMG_IDX}")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
labels_single = np.full(len(lvl_del), target_class, dtype=np.int64)

def plot_reliability(ax, probs_np: np.ndarray, labels_np: np.ndarray, title: str) -> None:
    confs = np.max(probs_np, axis=1)
    preds = np.argmax(probs_np, axis=1)
    accs  = (preds == labels_np).astype(float)
    bins  = np.linspace(0, 1, 11)
    bin_confs, bin_accs = [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (confs >= lo) & (confs < hi)
        if mask.sum() > 0:
            bin_confs.append(confs[mask].mean())
            bin_accs.append(accs[mask].mean())
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Perfect")
    ax.scatter(bin_confs, bin_accs, color="steelblue", s=60, zorder=5)
    if bin_confs:
        ax.plot(bin_confs, bin_accs, color="steelblue")
    ax.set_xlim(0, 1); ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy")
    ax.set_title(title); ax.legend(fontsize=8); ax.grid(alpha=0.3)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
plot_reliability(ax1, raw_all_probs, labels_single, "Raw model")
plot_reliability(ax2, cal_all_probs, labels_single, "Calibrated model")
plt.suptitle(f"Reliability diagram — Image {IMG_IDX} (26 perturbation steps)", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
ece_raw  = calculate_ece(raw_all_probs,  labels_single)
ece_cal  = calculate_ece(cal_all_probs,  labels_single)
tace_raw = calculate_tace(raw_all_probs, labels_single)
tace_cal = calculate_tace(cal_all_probs, labels_single)

print(f"{'Metric':<8} {'Raw':>10} {'Calibrated':>12} {'Delta':>10}")
print("-" * 44)
print(f"{'ECE':<8} {ece_raw:>10.4f} {ece_cal:>12.4f} {ece_cal - ece_raw:>+10.4f}")
print(f"{'TACE':<8} {tace_raw:>10.4f} {tace_cal:>12.4f} {tace_cal - tace_raw:>+10.4f}")
print("\nNegative delta = improvement (lower ECE/TACE is better)")
print("Note: single-image calibration overfits — Section 3 gives a more meaningful estimate.")

## Section 3 — Batch Evaluation (20 images)

Fit temperatures from all 20 images jointly, then evaluate per-image ECE/TACE to compute mean ± std.

In [ ]:
batch_logits_bins = {b: [] for b in range(NUM_BINS)}
batch_labels_bins = {b: [] for b in range(NUM_BINS)}

gen_del = BucketDeletionGenerator(num_buckets=25)

for i in tqdm(range(len(samples)), desc="Collecting logits"):
    tensor = img_batch[i:i+1]
    tc     = target_classes[i]
    sal    = torch.from_numpy(normalize_saliency(rise_sals[i]))

    seq_del_i, lvl_del_i = gen_del(tensor.cpu(), sal)

    with torch.no_grad():
        step_logits = raw_model(seq_del_i.to(device)).cpu().numpy()   # (26, 1000)

    for step_idx, logits in enumerate(step_logits):
        level   = lvl_del_i[step_idx]
        bin_idx = min(int(level * NUM_BINS), NUM_BINS - 1)
        batch_logits_bins[bin_idx].append(logits)
        batch_labels_bins[bin_idx].append(tc)

print(f"\n{'Bin':>4}  {'Level range':>12}  {'Samples':>8}")
print("-" * 30)
for b in range(NUM_BINS):
    lo, hi = b / NUM_BINS, (b + 1) / NUM_BINS
    print(f"{b:>4}  {lo:.1f}–{hi:.1f}        {len(batch_logits_bins[b]):>8}")

In [ ]:
batch_temps = []
for b in range(NUM_BINS):
    lgt = batch_logits_bins[b]
    lbl = batch_labels_bins[b]
    if len(lgt) > 1:
        T = temperature_scaling(np.stack(lgt), np.array(lbl, dtype=np.int64))
    else:
        T = 1.0
    batch_temps.append(round(T, 4))

global_recalx = ReCalXModel(raw_model, num_bins=NUM_BINS)
global_recalx.load_learned_temperatures(batch_temps)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(np.arange(NUM_BINS), batch_temps, color="steelblue", edgecolor="white", linewidth=0.5)
ax.axhline(1.0, color="red", linestyle="--", alpha=0.7, label="T=1 (no calibration)")
ax.set_xlabel("Perturbation bin")
ax.set_ylabel("Temperature T")
ax.set_title("Fitted temperatures — batch (20 images)")
ax.set_xticks(np.arange(NUM_BINS))
ax.set_xticklabels([f"{b/NUM_BINS:.1f}" for b in range(NUM_BINS)], rotation=45, fontsize=8)
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

print("Temperatures:", batch_temps)

In [ ]:
ece_raws, ece_cals   = [], []
tace_raws, tace_cals = [], []

for i in tqdm(range(len(samples)), desc="Evaluating images"):
    tensor = img_batch[i:i+1]
    tc     = target_classes[i]
    sal    = torch.from_numpy(normalize_saliency(rise_sals[i]))

    seq_del_i, lvl_del_i = gen_del(tensor.cpu(), sal)
    labels_i = np.full(len(lvl_del_i), tc, dtype=np.int64)

    _, _, raw_ap = evaluate_causal_metric(raw_model,    seq_del_i.to(device), lvl_del_i, tc)
    _, _, cal_ap = evaluate_causal_metric(global_recalx, seq_del_i.to(device), lvl_del_i, tc)

    ece_raws.append(calculate_ece(raw_ap,   labels_i))
    ece_cals.append(calculate_ece(cal_ap,   labels_i))
    tace_raws.append(calculate_tace(raw_ap,  labels_i))
    tace_cals.append(calculate_tace(cal_ap,  labels_i))

print(f"{'Metric':<8} {'Model':>12} {'Mean':>8} {'Std':>8}")
print("-" * 42)
for metric, raws, cals in [("ECE", ece_raws, ece_cals), ("TACE", tace_raws, tace_cals)]:
    print(f"{metric:<8} {'Raw':>12} {np.mean(raws):>8.4f} {np.std(raws):>8.4f}")
    print(f"{metric:<8} {'Calibrated':>12} {np.mean(cals):>8.4f} {np.std(cals):>8.4f}")
    print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (vals_raw, vals_cal, metric) in zip(axes, [
    (ece_raws,  ece_cals,  "ECE"),
    (tace_raws, tace_cals, "TACE"),
]):
    bplot = ax.boxplot(
        [vals_raw, vals_cal], labels=["Raw", "Calibrated"],
        patch_artist=True, widths=0.5,
        boxprops=dict(facecolor="lightblue"),
        medianprops=dict(color="red", linewidth=2),
    )
    ax.set_title(f"{metric} distribution  (N=20 images)")
    ax.set_ylabel(metric)
    ax.grid(axis="y", alpha=0.3)
    mr, sr = np.mean(vals_raw), np.std(vals_raw)
    mc, sc = np.mean(vals_cal), np.std(vals_cal)
    ax.text(0.05, 0.97,
            f"Raw: {mr:.3f}±{sr:.3f}\nCal: {mc:.3f}±{sc:.3f}",
            transform=ax.transAxes, va="top", fontsize=9,
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.6))

plt.suptitle("Calibration quality: Raw vs ReCalX  (global temperatures)", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
delta_ece = [r - c for r, c in zip(ece_raws, ece_cals)]   # positive = improvement
colors    = ["green" if d > 0 else "red" for d in delta_ece]

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(delta_ece)), delta_ece, color=colors, edgecolor="white", linewidth=0.5)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Image index")
ax.set_ylabel("ΔECE  (raw − calibrated)")
ax.set_title("Per-image ECE improvement from ReCalX  (positive = improved)")
ax.set_xticks(range(len(delta_ece)))
ax.grid(axis="y", alpha=0.3)
ax.legend(handles=[
    mpatches.Patch(color="green", label="Improved"),
    mpatches.Patch(color="red",   label="Worsened"),
])
plt.tight_layout(); plt.show()

n_improved = sum(d > 0 for d in delta_ece)
print(f"Improved: {n_improved}/{len(delta_ece)}  |  Mean ΔECE: {np.mean(delta_ece):+.4f}")

## Section 4 — Normalization Sanity Check

Different attribution methods produce saliency maps with very different value ranges. The bucket generators are rank-based (scale-invariant), so they work regardless, but the raw scale differences explain why some methods look broken when visualised without normalization.

In [ ]:
methods       = ["rise", "ig", "saliency", "gradientshap"]
method_labels = {"rise": "RISE", "ig": "IntegratedGradients",
                 "saliency": "Saliency", "gradientshap": "GradientShap"}

saliency_maps = {}
for method in tqdm(methods, desc="Running attributions"):
    sal = pipeline.generate_map(single_tensor, target_class, method).numpy()
    saliency_maps[method] = sal

print(f"\n{'Method':<22} {'Raw min':>10} {'Raw max':>10} {'Norm min':>10} {'Norm max':>10}")
print("-" * 66)
for m, sal in saliency_maps.items():
    sal_n = normalize_saliency(sal)
    print(f"{method_labels[m]:<22} {sal.min():>10.4f} {sal.max():>10.4f} {sal_n.min():>10.4f} {sal_n.max():>10.4f}")

print("\nNote: BucketDeletionGenerator uses argsort → rank-invariant, scale does not matter.")

In [ ]:
img_display = denormalize(single_tensor)
n_methods   = len(methods)

fig, axes = plt.subplots(2, n_methods + 1, figsize=(4 * (n_methods + 1), 7))

for row in range(2):
    axes[row, 0].imshow(img_display)
    axes[row, 0].axis("off")
    axes[row, 0].set_title("Original")

for col, method in enumerate(methods, start=1):
    sal_raw  = saliency_maps[method]
    sal_norm = normalize_saliency(sal_raw)

    # Top row: raw (clipped for display, but shows real scale in title)
    sal_disp = np.clip(sal_raw, np.percentile(sal_raw, 2), np.percentile(sal_raw, 98))
    sal_disp = (sal_disp - sal_disp.min()) / (sal_disp.max() - sal_disp.min() + 1e-8)
    axes[0, col].imshow(img_display)
    axes[0, col].imshow(sal_disp, cmap="jet", alpha=0.5)
    axes[0, col].axis("off")
    axes[0, col].set_title(
        f"{method_labels[method]}\nraw [{sal_raw.min():.3f}, {sal_raw.max():.3f}]",
        fontsize=8
    )

    # Bottom row: normalized [0, 1]
    axes[1, col].imshow(img_display)
    axes[1, col].imshow(sal_norm, cmap="jet", alpha=0.5)
    axes[1, col].axis("off")
    axes[1, col].set_title(f"{method_labels[method]}\nnormalized [0, 1]", fontsize=8)

axes[0, 0].set_ylabel("Raw (clipped display)", fontsize=9)
axes[1, 0].set_ylabel("Normalized [0, 1]", fontsize=9)
plt.suptitle(
    "Attribution maps: raw vs ReLU+min-max normalization\n"
    "(generators are rank-invariant, but visualization and Captum methods differ in scale)",
    fontsize=11
)
plt.tight_layout(); plt.show()